# Week 3: LLM Chat Assistant

本 Notebook 是暑期 AI 工程实践训练营第三周完整实验。

你将完成：

1. Prompt Engineering 基础练习。
2. Zero-shot 和 Few-shot Prompt 对比。
3. Hugging Face Pipeline 可选体验。
4. OpenAI 兼容 API 请求结构。
5. 没有 API Key 时使用 mock 模式继续开发。
6. Prompt Template。
7. 带历史记录的聊天助手。
8. 保存 Prompt 和对话记录。

## 1. 导入依赖并创建目录

In [ ]:
from pathlib import Path
import json
import os
import textwrap
from datetime import datetime

import requests
from dotenv import load_dotenv

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPT_DIR = ROOT_DIR / "prompts"
RESULT_DIR = ROOT_DIR / "results"
SRC_DIR = ROOT_DIR / "src"

for directory in [PROMPT_DIR, RESULT_DIR, SRC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT_DIR / ".env")

print("工程根目录:", ROOT_DIR)
print("Prompt 目录:", PROMPT_DIR)
print("结果目录:", RESULT_DIR)

## 2. Prompt Engineering 基础

Prompt 是你给大语言模型的任务说明。一个好的 Prompt 通常包含：

- 角色：让模型知道自己扮演什么角色。
- 任务：让模型知道要做什么。
- 上下文：提供必要背景。
- 输出格式：规定回答结构。
- 约束：说明不要做什么或注意什么。

In [ ]:
weak_prompt = "解释一下机器学习。"

strong_prompt = """
你是一名 AI 工程助教。
请面向大一到大二学生解释机器学习。

要求：
1. 用 3 个要点解释。
2. 每个要点不超过 50 字。
3. 最后给一个生活中的类比。
""".strip()

print("弱 Prompt:")
print(weak_prompt)
print("\n强 Prompt:")
print(strong_prompt)

观察：强 Prompt 比弱 Prompt 多了角色、对象、格式和约束，因此更容易得到可控输出。

## 3. Zero-shot 与 Few-shot

- Zero-shot：不给例子，直接让模型完成任务。
- Few-shot：给几个示例，让模型模仿格式和风格。

In [ ]:
zero_shot_prompt = """
请判断下面评论的情感是 positive、negative 还是 neutral。

评论：这个工具安装很简单，但是运行速度有点慢。
情感：
""".strip()

few_shot_prompt = """
请判断评论情感，只输出 positive、negative 或 neutral。

评论：这个软件非常好用，界面也清楚。
情感：positive

评论：安装失败，文档也看不懂。
情感：negative

评论：功能可以用，但没有特别惊喜。
情感：neutral

评论：这个工具安装很简单，但是运行速度有点慢。
情感：
""".strip()

print("Zero-shot Prompt:\n")
print(zero_shot_prompt)
print("\n" + "=" * 60 + "\n")
print("Few-shot Prompt:\n")
print(few_shot_prompt)

## 4. 保存 Prompt 示例

In [ ]:
prompt_examples = f"""
# Prompt Examples

## Weak Prompt

```text
{weak_prompt}
```

## Strong Prompt

```text
{strong_prompt}
```

## Zero-shot Prompt

```text
{zero_shot_prompt}
```

## Few-shot Prompt

```text
{few_shot_prompt}
```
""".strip()

prompt_path = PROMPT_DIR / "prompt_examples.md"
prompt_path.write_text(prompt_examples, encoding="utf-8")
print("Prompt 示例已保存:", prompt_path)

## 5. Hugging Face Pipeline 可选体验

这一部分会尝试加载一个很小的英文文本生成模型。如果下载失败，会自动使用 mock 函数继续运行。

In [ ]:
def mock_generate(prompt):
    return "[mock response] 这是一个模拟模型输出。真实项目中，这里会替换为本地模型或 API 返回。"


try:
    from transformers import pipeline
    generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")
    hf_output = generator("AI engineering is", max_new_tokens=30, do_sample=False)[0]["generated_text"]
    hf_mode = "transformers_pipeline"
except Exception as exc:
    hf_output = mock_generate("AI engineering is")
    hf_mode = "mock"
    print("Hugging Face 模型加载失败，使用 mock 输出。错误信息:", exc)

print("模式:", hf_mode)
print(hf_output)

## 6. OpenAI 兼容 API 请求结构

许多 LLM 服务都支持类似 OpenAI Chat Completions 的接口结构。常见请求包括：

- `model`：模型名称。
- `messages`：对话消息列表。
- `temperature`：控制回答随机性。

本 Notebook 支持：

- DeepSeek API。
- OpenAI 兼容 API。
- 无 API Key 时的 mock 模式。

In [ ]:
def get_api_config():
    if os.getenv("DEEPSEEK_API_KEY"):
        return {
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "base_url": os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
            "model": os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
            "provider": "deepseek",
        }
    if os.getenv("OPENAI_API_KEY"):
        return {
            "api_key": os.getenv("OPENAI_API_KEY"),
            "base_url": os.getenv("OPENAI_BASE_URL", "https://api.openai.com"),
            "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            "provider": "openai_compatible",
        }
    return None


api_config = get_api_config()
print("API 配置:", api_config if api_config else "未配置 API Key，将使用 mock 模式")

In [ ]:
def mock_chat_completion(messages):
    user_messages = [m["content"] for m in messages if m["role"] == "user"]
    latest_user_message = user_messages[-1] if user_messages else ""
    return {
        "provider": "mock",
        "model": "mock-llm",
        "content": "这是 mock 模式回答：我已经收到你的问题：" + latest_user_message,
    }


def call_chat_completion(messages, temperature=0.2):
    config = get_api_config()
    if config is None:
        return mock_chat_completion(messages)

    url = config["base_url"].rstrip("/") + "/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {config['api_key']}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": config["model"],
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    content = data["choices"][0]["message"]["content"]
    return {
        "provider": config["provider"],
        "model": config["model"],
        "content": content,
        "raw": data,
    }

## 7. 第一次 API 或 Mock 调用

In [ ]:
messages = [
    {"role": "system", "content": "你是一名耐心的 AI 工程助教。"},
    {"role": "user", "content": "请用三句话解释什么是 Prompt Engineering。"},
]

try:
    api_result = call_chat_completion(messages)
except Exception as exc:
    print("真实 API 调用失败，自动改用 mock 模式。错误信息:", exc)
    api_result = mock_chat_completion(messages)

print("Provider:", api_result["provider"])
print("Model:", api_result["model"])
print("Content:\n", api_result["content"])

api_result_path = RESULT_DIR / "api_response_example.json"
api_result_path.write_text(json.dumps(api_result, ensure_ascii=False, indent=2), encoding="utf-8")
print("API 示例结果已保存:", api_result_path)

## 8. Prompt Template

Prompt Template 用于把固定任务格式和动态变量组合起来。

In [ ]:
def build_explain_prompt(topic, audience, style):
    return f"""
你是一名 AI 工程助教。
请面向{audience}解释：{topic}

要求：
1. 使用{style}风格。
2. 输出 3 个要点。
3. 每个要点不超过 60 字。
4. 最后给一个小练习。
""".strip()


template_prompt = build_explain_prompt(
    topic="大语言模型中的 temperature 参数",
    audience="刚学完 Python 的本科生",
    style="清晰、具体、少术语",
)

print(template_prompt)

In [ ]:
template_messages = [
    {"role": "system", "content": "你是一名擅长教学的 AI 工程助教。"},
    {"role": "user", "content": template_prompt},
]

try:
    template_result = call_chat_completion(template_messages)
except Exception as exc:
    print("真实 API 调用失败，自动改用 mock 模式。错误信息:", exc)
    template_result = mock_chat_completion(template_messages)

template_output = template_result["content"]
print(template_output)

template_result_path = RESULT_DIR / "prompt_template_result.md"
template_result_path.write_text(template_output, encoding="utf-8")
print("Prompt Template 结果已保存:", template_result_path)

## 9. 构建带历史记录的聊天助手

聊天应用的关键是保存历史消息，并在每次请求时把历史一起发给模型。

In [ ]:
class ChatAssistant:
    def __init__(self, system_prompt):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]

    def ask(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        try:
            result = call_chat_completion(self.messages)
        except Exception as exc:
            print("真实 API 调用失败，自动改用 mock 模式。错误信息:", exc)
            result = mock_chat_completion(self.messages)
        assistant_message = result["content"]
        self.messages.append({"role": "assistant", "content": assistant_message})
        return assistant_message

    def save_history(self, path):
        payload = {
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "messages": self.messages,
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


assistant = ChatAssistant(
    system_prompt="你是一名 AI 工程实践训练营助教，回答要简洁、具体、可操作。"
)

answer1 = assistant.ask("什么是 Few-shot Prompt？")
print("Assistant 1:\n", answer1)

answer2 = assistant.ask("请基于刚才的解释，给一个情感分类的例子。")
print("\nAssistant 2:\n", answer2)

## 10. 保存聊天历史

In [ ]:
chat_history_path = RESULT_DIR / "chat_history.json"
assistant.save_history(chat_history_path)
print("聊天历史已保存:", chat_history_path)

## 11. 模拟流式输出

真实流式输出会逐步返回 token。这里用本地函数模拟这个效果，帮助理解流式输出的用户体验。

In [ ]:
def fake_stream(text, chunk_size=8):
    for i in range(0, len(text), chunk_size):
        yield text[i:i + chunk_size]


stream_text = "流式输出的核心体验是：用户不需要等完整回答生成完，模型可以边生成边显示。"
for chunk in fake_stream(stream_text):
    print(chunk, end="")

## 12. 最终检查

In [ ]:
expected_files = [
    PROMPT_DIR / "prompt_examples.md",
    RESULT_DIR / "chat_history.json",
    RESULT_DIR / "api_response_example.json",
    RESULT_DIR / "prompt_template_result.md",
]

for path in expected_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")

## 13. 学习复盘

运行完成后，请尝试回答：

1. Prompt Engineering 解决什么问题？
2. Zero-shot 和 Few-shot 有什么区别？
3. Prompt Template 为什么适合工程化？
4. OpenAI 兼容 API 请求里为什么要有 messages？
5. system、user、assistant 三种 role 分别代表什么？
6. 为什么聊天助手需要保存历史消息？
7. 没有 API Key 时，mock 模式对开发有什么帮助？
8. 如果继续改进这个助手，你会加什么功能？